# Kaismart Solutions S.A.S. — Transformación (Metodología Medallion)

**Parte 7 del ejercicio.** A partir de los problemas de calidad detectados en [`01_extraccion_y_eda.ipynb`](01_extraccion_y_eda.ipynb), se implementa el proceso de limpieza, transformación e integración siguiendo la metodología **Medallion**:

- **Bronze**: datos crudos, tal como llegan de las fuentes, solo con metadatos de trazabilidad de la ingesta.
- **Silver**: limpieza y estandarización **independiente** por fuente → `df_ventas_transformado`, `df_logistica_transformado`.
- **Gold**: integración de ambas fuentes por `pedido_id` + datamarts listos para análisis de negocio.

Cada decisión de imputación/estandarización está documentada en el código (`src/kaismart_etl/transform/silver.py`) y se resume aquí mediante los `reporte` que retorna cada función — no se realiza ninguna imputación automática sin justificar el motivo.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from kaismart_etl import quality
from kaismart_etl.extract import extract_logistica, extract_ventas
from kaismart_etl.transform import bronze, gold, silver

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

df_ventas = extract_ventas()
df_logistica = extract_logistica()
print(f"df_ventas: {df_ventas.shape}  |  df_logistica: {df_logistica.shape}")

## Capa Bronze

Se persiste una copia inmutable de los datos crudos (sin ninguna limpieza), agregando solo metadatos de trazabilidad (`_fuente`, `_fecha_ingesta`). Esto permite reprocesar Silver/Gold en el futuro sin necesidad de volver a consultar las fuentes originales.

In [ ]:
ruta_bronze_ventas = bronze.guardar_bronze(df_ventas, "ventas", fuente="mysql.clientes.ventas")
ruta_bronze_logistica = bronze.guardar_bronze(
    df_logistica, "logistica", fuente="kaismart_eventos_logisticos.xlsx"
)
print("Bronze ventas:", ruta_bronze_ventas)
print("Bronze logistica:", ruta_bronze_logistica)

## Capa Silver — `df_ventas_transformado`

Decisiones de limpieza aplicadas (ver `silver.limpiar_ventas`):

1. Se eliminan filas **completamente duplicadas** y duplicados de `id_venta` (llave primaria) — conservando el primer registro.
2. Filas sin `pedido_id` se descartan (sin esa llave no es posible integrar con logística).
3. `fecha_venta` se parsea a `datetime`; fechas inválidas quedan como `NaT` (no se inventa una fecha).
4. `ciudad`, `canal`, `categoria` se **estandarizan** contra un catálogo controlado (mayúsculas/tildes/espacios).
5. `metodo_pago` nulo → `"NO INFORMADO"` (sin significado de negocio claro para el nulo).
6. `cantidad` y `precio_unitario` inválidos (≤ 0) → se imputan con la **mediana por categoría** (variables con outliers, more robusta que la media).
7. `valor_descuento` nulo → se imputa con **0** (representa "sin descuento", no una estimación).
8. `valor_bruto` y `valor_neto` se **recalculan** a partir de `cantidad`, `precio_unitario` y `valor_descuento` para garantizar consistencia aritmética.
9. `calificacion_cliente`: los nulos **se conservan** (el cliente no calificó, es válido); solo se corrigen valores fuera de rango 1-5.

In [ ]:
df_ventas_transformado, reporte_ventas = silver.limpiar_ventas(df_ventas)

print("Reporte de limpieza df_ventas -> df_ventas_transformado:")
for k, v in reporte_ventas.items():
    print(f"  {k}: {v}")

print(f"\nShape antes: {df_ventas.shape}  ->  Shape despues: {df_ventas_transformado.shape}")
df_ventas_transformado.head()

## Capa Silver — `df_logistica_transformado`

Decisiones de limpieza aplicadas (ver `silver.limpiar_logistica`):

1. Se eliminan filas completamente duplicadas y duplicados de `evento_id` (conservando el primero).
2. Eventos sin `pedido_id` se descartan (no se pueden vincular a ninguna venta).
3. `fecha_evento` se parsea a `datetime`; inválidas quedan como `NaT`.
4. `estado_evento`, `ciudad_destino`, `transportadora` se estandarizan contra catálogos controlados.
5. `incidencia` nulo → `"SIN INCIDENCIA"` (el nulo aquí tiene un significado de negocio real: no ocurrió ningún incidente).
6. `costo_envio`: se limpia el formato de texto tipo `"$ 45.000"` y se convierte a numérico; los nulos resultantes se imputan con la **mediana por transportadora** (variable monetaria con outliers).
7. `tiempo_etapa_horas` negativo (error de captura) → se lleva a `NaN` y se imputa con la **mediana por estado_evento** (cada etapa tiene una duración típica distinta).

In [ ]:
df_logistica_transformado, reporte_logistica = silver.limpiar_logistica(df_logistica)

print("Reporte de limpieza df_logistica -> df_logistica_transformado:")
for k, v in reporte_logistica.items():
    print(f"  {k}: {v}")

print(f"\nShape antes: {df_logistica.shape}  ->  Shape despues: {df_logistica_transformado.shape}")
df_logistica_transformado.head()

### Validación: perfil de nulos antes vs. después de Silver

Se compara el perfil de nulos original contra el transformado para confirmar que las columnas tratadas (según lo documentado arriba) efectivamente mejoraron, y que las que se dejaron nulas a propósito (`calificacion_cliente`, fechas inválidas) siguen siendo nulas.

In [ ]:
print("Nulos ANTES (df_ventas):")
display(quality.perfil_nulos(df_ventas))
print("\nNulos DESPUES (df_ventas_transformado):")
display(quality.perfil_nulos(df_ventas_transformado))

print("\nNulos ANTES (df_logistica):")
display(quality.perfil_nulos(df_logistica))
print("\nNulos DESPUES (df_logistica_transformado):")
display(quality.perfil_nulos(df_logistica_transformado))

# Persistencia de la capa Silver
df_ventas_transformado.to_parquet(PROJECT_ROOT / "data/silver/ventas_transformado.parquet", index=False)
df_logistica_transformado.to_parquet(PROJECT_ROOT / "data/silver/logistica_transformado.parquet", index=False)

## Capa Gold — integración por `pedido_id` + datamarts

`df_logistica_transformado` tiene múltiples eventos por `pedido_id`; se colapsan a una fila resumen por pedido (estado actual, transportadora, ciclo logístico total, costo de envío, si tuvo incidencia) y se integra con `df_ventas_transformado` mediante un `merge` por `pedido_id`.

In [ ]:
df_gold_pedidos, reporte_gold = gold.construir_gold_pedidos(
    df_ventas_transformado, df_logistica_transformado
)
print("Reporte de integracion Gold:")
for k, v in reporte_gold.items():
    print(f"  {k}: {v}")

gold.guardar_gold(df_gold_pedidos, "pedidos_360")

marts = gold.construir_marts(df_gold_pedidos)
for nombre_mart, df_mart in marts.items():
    gold.guardar_gold(df_mart, nombre_mart)
    print(f"\n--- Mart: {nombre_mart} ---")
    display(df_mart.head(10))

df_gold_pedidos.head()

## Parte 8. Automatización

Todo este flujo (extracción → Bronze → Silver → Gold) está encapsulado en `kaismart_etl.pipeline.run_etl()`. La automatización periódica con la librería `schedule` está implementada en `kaismart_etl/scheduler.py` y se ejecuta con:

```bash
python scripts/run_scheduler.py
```

Esto corre el pipeline inmediatamente y luego cada `SCHEDULE_INTERVAL_MINUTES` minutos (configurable en `.env`), de forma indefinida, hasta detenerlo con `Ctrl+C`.